# A sweep from a notebook

A thin example. Everything here is one of the package's commands, called from Python, so the notebook writes nothing
the commands would not write and reads only files they wrote. It never opens a file that judges a choice (a test
file, a reveal file): `test-report` and `reveal-report` read those, from the command line, after `select` has
recorded the choice and a RULE covers it (docs/HOW_WE_WORK.md, CLAUDE.md rule 3). docs/WALKTHROUGH.md is the same
result step by step.

Use an environment from START_HERE.md as the kernel (`.venv-main`), and open the notebook from the repository's
`notebooks/` folder.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SWEEP_FILE = "sweeps/examples/cluster_kmeans.json"
SWEEP_ID = "example_cluster_kmeans"


def cli(*args, show=40):
    """Run one sweepkit command from the repository folder, as from a terminal, and show the end of its output."""
    env = dict(os.environ, PYTHONPATH=str(REPO / "src"))
    done = subprocess.run([sys.executable, "-m", "sweepkit.cli", *args], cwd=REPO, env=env, capture_output=True,
                          text=True)
    print("\n".join((done.stdout + done.stderr).splitlines()[-show:]))
    done.check_returncode()

## What will run

`check` expands the sweep file into jobs without running anything. The same expansion from Python:

In [ ]:
sys.path.insert(0, str(REPO / "src"))
from sweepkit import catalog
from sweepkit.sweep.design import expand

catalog.load_all()
jobs, summary = expand(json.loads((REPO / SWEEP_FILE).read_text(encoding="utf-8")))
print(summary["configs"], "configurations,", summary["jobs"], "jobs")
for stage, label in sorted({(job["stage"], job["label"]) for job in jobs}):
    print(f"  {stage}: {label}")

## Run it, and compare the configurations on the label-free criteria

A result that will be judged later (revealed, tested, reported) needs its RULE fixed before this run
(docs/HOW_WE_WORK.md): `rule-fix` from the command line first (docs/WALKTHROUGH.md, step 4), then this cell. The
code checks it: `rule-fix` asks for a reason (`--late-reason`) once runs of the sweep exist, and `reveal-report`
refuses a choice recorded before its RULE. This example is for learning and runs without one.

`run` is resumable: run the cell again and finished jobs are skipped. `analyze` reads the choosing files only.

In [ ]:
cli("run", "--sweep-file", SWEEP_FILE, "--device", "cpu", "--max-consecutive-failures", "3", show=3)
cli("analyze", "--sweep", SWEEP_ID, "--confidence", "0.95", "--n-boot", "2000", "--seed", "1", "--meaningful",
    "0.75", "--top", "10", show=4)

## The analysis as a table

The newest analysis report of the sweep, as `analyze` wrote it (`artifacts/analysis/<sweep>_<utc>/report.json`):

In [ ]:
import pandas as pd

folder = sorted((REPO / "artifacts" / "analysis").glob(f"{SWEEP_ID}_*"))[-1]
report = json.loads((folder / "report.json").read_text(encoding="utf-8"))
criterion = report["select"]["metric"]
table = pd.DataFrame([{"configuration": entry["label"], "hash": entry["config_hash"][:12],
                       criterion: entry["value"], "low": entry["interval"]["low"], "high": entry["interval"]["high"],
                       "clusters": entry["metrics"].get("n_clusters"),
                       "stability (ARI)": entry["stability"]["value"]} for entry in report["ranked"]])
print(folder.name)
table

## Next

The choice is a command, so it is recorded with its reason and time:

```
python -m sweepkit.cli select --sweep example_cluster_kmeans --config <hash> --name kmeans_best --reason "..."
```

`reveal-report` refuses unless a RULE covering the sweep was fixed before the choice was recorded; for a judged
result it is fixed before the run (above). Then `reveal-report` and `export`, from the command line
(docs/WALKTHROUGH.md, steps 9 and 10).